### Step 1: Download and Load the Data
We will extract the file ID from the provided Google Drive link, download it as an Excel file, and load it into a pandas DataFrame.

In [1]:
import pandas as pd
import requests

# Google Drive link file ID
file_id = '119DwKkes8OjM6pBuQ2gDGuYpYYegXG7u'
download_url = f'https://docs.google.com/spreadsheets/d/{file_id}/export?format=xlsx'

# Download and read into pandas
response = requests.get(download_url)
with open('data.xlsx', 'wb') as f:
    f.write(response.content)

# Load the Excel sheet
df = pd.read_excel('data.xlsx')

# Clean column names to remove whitespace
df.columns = df.columns.str.strip()
display(df.head())

,Fecha,Laboratorio,Cultivo,Planta,Time,7in1_Nitrogen[mg/kg],7in1_Phosphorus[mg/kg],7in1_Potasium[mg/kg],7in1_Ph[pH],7in1_Moisture[%RH],...,Pynamometer_Radiation[W/m^2],Air_sensor_Temperature[C],Air_sensor_Humidity[%RH],RGB_capture,Tapo_capture,Parrot_capture,Posicion_x,Posicion_y,Tratamiento,Tratamiento iniciado
0,2024-09-11,Quindio,Tomatoes,121,2024-09-11 11:50:00,522.0,644.0,1214.0,6.30,36.9,...,24.6,29.0,52.5,1,1,5,1.0,12.0,T5_N0P0K2,Si
1,2024-09-11,Quindio,Tomatoes,145,2024-09-11 11:53:00,272.0,373.0,693.0,9.00,29.6,...,28.2,28.7,52.9,1,1,5,1.0,14.4,T6_N0P1K1,Si
2,2024-09-11,Quindio,Tomatoes,165,2024-09-11 11:58:00,161.0,228.0,451.0,6.33,20.4,...,26.1,28.5,56.5,1,1,5,3.2,15.6,T8_N0P2K1,Si
3,2024-09-11,Quindio,Tomatoes,172,2024-09-11 12:24:00,140.0,198.0,397.0,6.40,23.0,...,30.4,29.7,52.6,1,1,5,3.2,16.8,T11_N1P1K0,Si
4,2024-08-14,Quindio,Tomatoes,73,2024-08-14 10:24:00,95.0,140.0,288.0,6.50,26.1,...,28.9,25.8,66.3,1,1,5,1.0,7.2,T19_N2P0K0,Si


### Step 2: Count the Treatments
We will now count the occurrences of each unique treatment in the `tratamiento` column and identify the one with the highest count.

In [2]:
if 'Tratamiento' in df.columns:
    treatment_counts = df['Tratamiento'].value_counts().reset_index()
    treatment_counts.columns = ['Tratamiento', 'count']

    print("--- Treatment Counts ---")
    display(treatment_counts)

    highest_treatment = treatment_counts.iloc[0]['Tratamiento']
    highest_count = treatment_counts.iloc[0]['count']
    print(f"\nThe treatment with the highest amount of associated data is '{highest_treatment}' with {highest_count} rows.")
else:
    print("Error: 'Tratamiento' column not found in the dataset. Available columns:", df.columns.tolist())

--- Treatment Counts ---


,Tratamiento,count
0,T21_N2P2K0,43
1,T24_N2P1K1,42
2,T15_N1P1K1,42
3,T9_N0P2K2,36
4,T7_N0P1K2,36
5,T30_BIO N2P2K2,35
6,T26_N2P2K1,35
7,T13_N1P0K1,35
8,T10_N1P0K0,35
9,T22_N2P0K1,34



The treatment with the highest amount of associated data is 'T21_N2P2K0' with 43 rows.


### Step 3: Interactive Table
Let's create an interactive widget using `ipywidgets` so you can select any treatment and view its corresponding data rows dynamically.

In [3]:
import ipywidgets as widgets
from IPython.display import display, clear_output

if 'Tratamiento' in df.columns:
    treatments = sorted(df['Tratamiento'].dropna().unique().tolist())

    dropdown = widgets.Dropdown(
        options=treatments,
        description='Treatment:',
        disabled=False,
    )

    output = widgets.Output()

    def on_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            with output:
                clear_output(wait=True)
                selected_treatment = change['new']
                filtered_df = df[df['Tratamiento'] == selected_treatment]
                print(f"Displaying {len(filtered_df)} rows for treatment: {selected_treatment}")
                display(filtered_df)

    dropdown.observe(on_change, names='value')

    # Display dropdown and initial selection
    display(dropdown)
    display(output)

    # Trigger initial display
    with output:
        initial_treatment = dropdown.value
        filtered_df = df[df['Tratamiento'] == initial_treatment]
        print(f"Displaying {len(filtered_df)} rows for treatment: {initial_treatment}")
        display(filtered_df)
else:
    print("Cannot build interactive table without 'Tratamiento' column.")

Dropdown(description='Treatment:', options=('T10_N1P0K0', 'T11_N1P1K0', 'T12_N1P2K0', 'T13_N1P0K1', 'T14_N1P0K…

Output()

In [4]:
import re
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Extract the NPK part of the Treatment
def extract_npk(treatment):
    if not isinstance(treatment, str):
        return None
    # Use regular expression to find N, P, and K with their corresponding numbers
    match = re.search(r'N\d+P\d+K\d+', treatment)
    return match.group(0) if match else None

# Apply extraction
df['NPK_part'] = df['Tratamiento'].apply(extract_npk)

# 2. Count the occurrences of NPK
npk_counts = df['NPK_part'].value_counts().reset_index()
npk_counts.columns = ['NPK_Configuration', 'count']

print("--- NPK Part Counts ---")
display(npk_counts)

highest_npk = npk_counts.iloc[0]['NPK_Configuration']
highest_npk_count = npk_counts.iloc[0]['count']
print(f"\nThe NPK configuration with the highest amount of associated data is '{highest_npk}' with {highest_npk_count} rows.\n")

# 3. Setup Interactive Dropdown Filter
npk_list = sorted(df['NPK_part'].dropna().unique().tolist())

npk_dropdown = widgets.Dropdown(
    options=npk_list,
    description='NPK Config:',
    disabled=False,
)

npk_output = widgets.Output()

def on_npk_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        with npk_output:
            clear_output(wait=True)
            selected_npk = change['new']
            filtered_df = df[df['NPK_part'] == selected_npk]
            print(f"Displaying {len(filtered_df)} rows for NPK: {selected_npk}")
            display(filtered_df)

npk_dropdown.observe(on_npk_change, names='value')

# Display the dropdown and initial selection output
display(npk_dropdown)
display(npk_output)

with npk_output:
    initial_npk = npk_dropdown.value
    filtered_df = df[df['NPK_part'] == initial_npk]
    print(f"Displaying {len(filtered_df)} rows for NPK: {initial_npk}")
    display(filtered_df)

--- NPK Part Counts ---


,NPK_Configuration,count
0,N1P1K1,75
1,N2P2K2,68
2,N0P0K0,67
3,N2P2K0,43
4,N2P1K1,42
5,N0P2K2,36
6,N0P1K2,36
7,N2P2K1,35
8,N1P0K1,35
9,N1P0K0,35



The NPK configuration with the highest amount of associated data is 'N1P1K1' with 75 rows.



Dropdown(description='NPK Config:', options=('N0P0K0', 'N0P0K1', 'N0P0K2', 'N0P1K0', 'N0P1K1', 'N0P1K2', 'N0P2…

Output()